# News2Stock Analyser 데이터 준비
뉴스 기사를 전달하면 긍/부정 분석 뿐 아니라 특정 주식에 대한 긍/부정 평가를 하는 모델 구현

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

## 데이터 준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [ ]:
%pip install datasets openai

In [4]:
from datasets import load_dataset 

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [6]:
# train dataset에서 economy 뉴스만 필터링
economy_dataset = dataset['train'].filter(lambda row: row['category'] == 'economy')
print(len(economy_dataset))
economy_dataset[0]

Filter:   0%|          | 0/22194 [00:00<?, ? examples/s]

17088


{'date': '2022-07-03 17:14:37',
 'category': 'economy',
 'press': 'YTN ',
 'title': '추경호 중기 수출지원 총력 무역금융 40조 확대',
 'document': '앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
 'link': 'https://n.news.naver.com/mne

In [7]:
# DataFrame 변환
df = economy_dataset.to_pandas()
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## sLLM 답변 데이터 생성
LLM을 이용해서 sLLMㅇ 답변했으면 하는 내용을 생성해낸다.
이 때 답변 품질이 중요하므로, 되도록 상위 모델을 사용하는 것이 좋다.

In [8]:
# 구조화 된 답변 준비
from pydantic import BaseModel, Field
from typing import List 

class StockAnalysis(BaseModel):
    """경제 뉴스 분석 결과 클래스"""

    stock_related: bool = Field(description='뉴스와 주식 종목간의 연관성 여부')
    summary: str = Field(description='뉴스 요약')

    positive_stocks: List[str] = Field(
        default_factory=list,
        description='긍정적 영향이 예상 되는 주식 종목명 목록'
    )

    positive_keywords: List[str] = Field(
        default_factory=list,
        description='긍정적 영향의 근거가 되는 키워드 목록'
    )

    positive_reasons: str = Field(
        description='긍정적 영향이 예상 되는 이유'
    )

    negative_stocks: List[str] = Field(
        default_factory=list,
        description='부정적 영향이 예상 되는 주식 종목명 목록'
    )

    negative_keywords: List[str] = Field(
        default_factory=list,
        description='부정적 영향의 근거가 되는 키워드 목록'
    )

    negative_reasons: str = Field(
        description='부정적 영향이 예상 되는 이유'
    )

In [9]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,
이유/근거 등을 분석하는 금융 분석 전문가입니다.

다음 출력 지시사항을 지켜주세요.
1. 뉴스와 종목간의 영향성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 영향성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정 영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
"""

user_prompt = """
다음 뉴스 기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', user_prompt)
])

# 첫 번째 뉴스 기사
news = df['document'][0]
prompt.invoke({'news' : news})

ChatPromptValue(messages=[SystemMessage(content="\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는 긍정/부정 영향 여부,\n이유/근거 등을 분석하는 금융 분석 전문가입니다.\n\n다음 출력 지시사항을 지켜주세요.\n1. 뉴스와 종목간의 영향성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 영향성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정 영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정 영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='\n다음 뉴스 기사의 내용에 대해 심층적인 분석을 수행해주세요.\n\n[news]\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니

In [10]:
# llm 체인 생성
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4.1')

chain = prompt | llm.with_structured_output(StockAnalysis)

def analyze_news(news):
    return chain.invoke({'news' : news})

analyze_news(news)

StockAnalysis(stock_related=True, summary='정부가 하반기 수출 확대를 위해 무역금융 확대(40조 원 이상 추가), 물류비 지원, 임시선박 투입 등 수출 중소·중견기업 지원 대책을 발표했습니다. 이를 통해 국제 해상운임 불안, 원자재 가격 상승 등 대외 리스크를 완화하고, 반도체 등 첨단산업 수출 활성화 및 무역수지 개선을 목표로 하고 있습니다.', positive_stocks=['현대글로비스', '팬오션', '대한해운', '삼성전자', 'SK하이닉스', 'CJ대한통운', '동국제강', '포스코홀딩스'], positive_keywords=['무역금융 확대', '수출 기업 지원', '임시선박 투입', '반도체 첨단산업 육성', '물류비 지원', '해외 전시회 지원'], positive_reasons='대규모 무역금융 확대와 정부의 물류 지원, 첨단 산업 육성 정책은 수출 비중이 높은 반도체(삼성전자, SK하이닉스) 및 철강(포스코홀딩스, 동국제강), 그리고 해운&물류(현대글로비스, 팬오션, 대한해운, CJ대한통운) 기업에 성장 기회와 비용 절감 효과를 제공할 것으로 예상됩니다.', negative_stocks=[], negative_keywords=[], negative_reasons='')

In [11]:
news = df['document'][100]
analyze_news(news)

StockAnalysis(stock_related=True, summary="공유수면 점용·사용 허가 시 이해관계자의 의견을 반드시 수렴하도록 한 '공유수면 관리 및 매립에 관한 법률' 개정안이 시행됐다. 해상풍력, 관광시설 등 공유수면 이용 사업의 경우, 앞으로 어업인 등 이해 당사자 의견 청취와 공고 의무가 강화된다. 이는 사회적 갈등 해소 목적이지만, 사업 추진 절차가 추가되고 허가 지연 가능성이 높아질 전망이다.", positive_stocks=[], positive_keywords=[], positive_reasons='', negative_stocks=['씨에스윈드', '태경케미컬', '한화솔루션', '한국조선해양'], negative_keywords=['해상풍력', '공유수면 점용', '허가 지연', '이해관계자 의견수렴'], negative_reasons='해상풍력 및 공유수면 점용·사용 관련 대규모 개발사업의 경우, 이해관계자(어업인 등)의 의견 청취 및 공고 의무 강화로 인허가 과정이 복잡해지고 사업 지연 또는 비용 증가가 예상된다. 이에 해상풍력 및 관련 설비 제조·시공업체에게 단기적으로 부정적 영향이 우려된다.')

In [12]:
import pandas as pd

# 1000건의 데이터 샘플링 -> LLM 질의
df = df[:1000]
df['content'] = df['title'] + '\n' + df['document']

news_df = df['content']
print(len(news_df))
news_df.head()

1000


0    추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경...
1    해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑...
2    에디슨이노 이승훈 제이스페이스 대표 사내이사 선임\n기사내용 요약 우주발사체 사업 ...
3    SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차\nSK바이오사이언...
4    금융당국 “증시 변동성 완화 조치”\n4일부터 석달간 증권사 신용융자담보비율 유지의...
Name: content, dtype: str

In [13]:
# 응답 생성
from tqdm import tqdm 

results = []

for content in tqdm(news_df):
    result = analyze_news(content)
    results.append(result)

100%|██████████| 1000/1000 [1:03:43<00:00,  3.82s/it]


In [14]:
# 응답 저장
df['result'] = results 
df.head()

,date,category,press,title,document,link,summary,content,result
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ...",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경...,stock_related=True summary='정부가 수출 확대를 위해 중소·중...
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑...,"stock_related=True summary=""인터컨티넨탈 서울 코엑스의 뷔페 ..."
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임\n기사내용 요약 우주발사체 사업 ...,stock_related=True summary='에디슨이노가 사명을 이노시스로 변...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차\nSK바이오사이언...,stock_related=True summary='SK바이오사이언스가 글로벌 사업 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...,금융당국 “증시 변동성 완화 조치”\n4일부터 석달간 증권사 신용융자담보비율 유지의...,stock_related=True summary='금융당국이 최근 코스피 지수 하락...


In [15]:
# json 변환
import json
# - pydantic.BaseModel.model_dump() -> dict
# - pydantic.BaseModel.model_dump_json() -> json_str 

# json 파싱
def parse_to_json(obj):
    return obj.model_dump_json()

df['result_json'] = df['result'].apply(parse_to_json)
df.head()

,date,category,press,title,document,link,summary,content,result,result_json
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ...",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경...,stock_related=True summary='정부가 수출 확대를 위해 중소·중...,"{""stock_related"":true,""summary"":""정부가 수출 확대를 위해..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑...,"stock_related=True summary=""인터컨티넨탈 서울 코엑스의 뷔페 ...","{""stock_related"":true,""summary"":""인터컨티넨탈 서울 코엑스..."
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임\n기사내용 요약 우주발사체 사업 ...,stock_related=True summary='에디슨이노가 사명을 이노시스로 변...,"{""stock_related"":true,""summary"":""에디슨이노가 사명을 이노..."
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차\nSK바이오사이언...,stock_related=True summary='SK바이오사이언스가 글로벌 사업 ...,"{""stock_related"":true,""summary"":""SK바이오사이언스가 글로..."
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...,금융당국 “증시 변동성 완화 조치”\n4일부터 석달간 증권사 신용융자담보비율 유지의...,stock_related=True summary='금융당국이 최근 코스피 지수 하락...,"{""stock_related"":true,""summary"":""금융당국이 최근 코스피 ..."


In [16]:
# 결측치 제거
df = df.dropna(subset=['result_json'])
df = df.reset_index(drop=True)
df.head()

,date,category,press,title,document,link,summary,content,result,result_json
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ...",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경...,stock_related=True summary='정부가 수출 확대를 위해 중소·중...,"{""stock_related"":true,""summary"":""정부가 수출 확대를 위해..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑...,"stock_related=True summary=""인터컨티넨탈 서울 코엑스의 뷔페 ...","{""stock_related"":true,""summary"":""인터컨티넨탈 서울 코엑스..."
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임\n기사내용 요약 우주발사체 사업 ...,stock_related=True summary='에디슨이노가 사명을 이노시스로 변...,"{""stock_related"":true,""summary"":""에디슨이노가 사명을 이노..."
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차\nSK바이오사이언...,stock_related=True summary='SK바이오사이언스가 글로벌 사업 ...,"{""stock_related"":true,""summary"":""SK바이오사이언스가 글로..."
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...,금융당국 “증시 변동성 완화 조치”\n4일부터 석달간 증권사 신용융자담보비율 유지의...,stock_related=True summary='금융당국이 최근 코스피 지수 하락...,"{""stock_related"":true,""summary"":""금융당국이 최근 코스피 ..."


## 학습용 데이터셋 변환
- system
- user(human)
- assistant

In [17]:
df['system'] = system_prompt 

df = df.rename(columns={
    'content' : 'user',
    'result_json' : 'assistant'
})

df[['system', 'user', 'assistant']]

,system,user,assistant
0,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경...,"{""stock_related"":true,""summary"":""정부가 수출 확대를 위해..."
1,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑...,"{""stock_related"":true,""summary"":""인터컨티넨탈 서울 코엑스..."
2,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",에디슨이노 이승훈 제이스페이스 대표 사내이사 선임\n기사내용 요약 우주발사체 사업 ...,"{""stock_related"":true,""summary"":""에디슨이노가 사명을 이노..."
3,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차\nSK바이오사이언...,"{""stock_related"":true,""summary"":""SK바이오사이언스가 글로..."
4,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",금융당국 “증시 변동성 완화 조치”\n4일부터 석달간 증권사 신용융자담보비율 유지의...,"{""stock_related"":true,""summary"":""금융당국이 최근 코스피 ..."
...,...,...,...
995,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",1년 넘게 입출금 없는 은행 예금 15조 8천억 원…“전면 재점검해 사고 예방해야”...,"{""stock_related"":true,""summary"":""국내 4대 은행(KB국민..."
996,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",SGA 한국교육학술정보원과 204억원 규모 나이스 구축 공급계약 체결\nSGA는 한...,"{""stock_related"":true,""summary"":""SGA가 한국교육학술정보..."
997,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",“갯벌의 숨은 매력 사진에 담아주세요”\n해수부·해양환경공단 한국의 갯벌 가치 알린...,"{""stock_related"":false,""summary"":""해양수산부와 해양환경공..."
998,"\n당신은 금융/경제 뉴스의 핵심 내용을 요약해 설명하고, 특정 상장 종목에 미치는...",더뉴스 월세 60% 시대·매수 심리 뚝...부동산 시장 변화는\n진행 김영수 앵커 ...,"{""stock_related"":true,""summary"":""전국의 월세 거래 비중이..."


In [18]:
# json 파일 출력
df[['system', 'user', 'assistant']].to_json(
    'train.json',           # 파일명
    orient='records',       # dataframe 각 행이 하나의 json 객체로 저장
    force_ascii=False,      # 비영어권문자 ascii변환 안함
    indent=4                # 들여쓰기
)

In [19]:
# hugging face dataset upload
from datasets import Dataset
import os

dataset = Dataset.from_pandas(df[['system', 'user', 'assistant']])
dataset.push_to_hub(
    'blimu/naver-economy-news2stock2',
    token=os.environ['HF_TOKEN']
)

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/blimu/naver-economy-news2stock2/commit/0c58e9d51e391e203cbf2a0f813020525256402f', commit_message='Upload dataset', commit_description='', oid='0c58e9d51e391e203cbf2a0f813020525256402f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/blimu/naver-economy-news2stock2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='blimu/naver-economy-news2stock2'), pr_revision=None, pr_num=None)